# Task 1: Data Quality & Leakage Audit
**Notebook Name:** `01_exploratory_analysis.ipynb`  
**Dataset:** [AI4I 2020 Predictive Maintenance Dataset](https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset)  

### Objective:
Perform a rigorous data quality assessment and data-leakage audit to establish safe boundaries for predictive modeling.

In [ ]:
import pandas as pd
import numpy as np

# URL to the AI4I 2020 Predictive Maintenance Dataset on UCI
url = 'https://archive.ics.uci.edu/static/public/601/ai4i+2020+predictive+maintenance+dataset.zip'

# Load dataset
df = pd.read_csv(url)
print(f"Loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns.")
display(df.head())

### 1. Data Quality Check
Let's verify null values, data types, and unique counts across all attributes to audit data integrity.

In [ ]:
# Check missing values and data types
info_df = pd.DataFrame({
    'Data Type': df.dtypes,
    'Missing Values': df.isnull().sum(),
    'Unique Values': df.nunique()
})
display(info_df)

### 2. Data Leakage Audit

To ensure our model remains valid for real-world scenarios, we must audit columns that might leak information:
- **Identifiers (`UDI`, `Product ID`):** High cardinality columns with unique or sequential values. They do not hold physical significance and must be dropped to prevent overfitting.
- **Failure Mode Labels (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`):** These labels are direct components or causes used to define our prediction target (`Machine failure`). Including them as features would introduce fatal target leakage (e.g. if any failure mode is 1, machine failure is almost certainly 1).

In [ ]:
# Check target leakage: Correlation of specific failure modes with 'Machine failure'
failure_modes = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

print("--- Failure Mode Association with Target (Machine failure) ---")
for mode in failure_modes:
    contingency = pd.crosstab(df[mode], df['Machine failure'])
    print(f"
Contingency table for {mode} vs Machine failure:")
    print(contingency)

# High correlation indicates strong mapping which would cause leakage if included
correlation = df[failure_modes + ['Machine failure']].corr()['Machine failure']
print("
--- Direct Correlations with Prediction Target ---")
print(correlation)

### 3. Feature Selection Decision

Based on our audit, we establish the following feature selection boundaries:

| Column | Status | Reason |
| :--- | :--- | :--- |
| `UDI` | **Excluded** | Arbitrary sequential identifier. No machine correlation. |
| `Product ID` | **Excluded** | Product variant identification string; redundant with `Type`. |
| `TWF` | **Excluded** | Target-revealing failure mode label (Target Leakage). |
| `HDF` | **Excluded** | Target-revealing failure mode label (Target Leakage). |
| `PWF` | **Excluded** | Target-revealing failure mode label (Target Leakage). |
| `OSF` | **Excluded** | Target-revealing failure mode label (Target Leakage). |
| `RNF` | **Excluded** | Target-revealing failure mode label (Target Leakage). |
| `Type` | **Included** | Indicates quality variant (L/M/H), which affects tool wear bounds. |
| `Air temperature [K]` | **Included** | Direct physical measurement. |
| `Process temperature [K]`| **Included** | Direct physical measurement. |
| `Rotational speed [rpm]` | **Included** | Direct physical measurement. |
| `Torque [Nm]` | **Included** | Direct physical measurement. |
| `Tool wear [min]` | **Included** | Direct physical measurement. |
| `Machine failure` | **Target** | Binary classification goal target variable. |

We will enforce this clean feature split in our modeling pipelines.